# Semantic Retrieval Demo

This notebook demonstrates how to:
1. Extract sentences from MD files with source tracking
2. Generate embeddings using Qwen3-Embedding:8B via Ollama
3. Store embeddings in FAISS for fast similarity search
4. Perform semantic search with cosine similarity
5. Synthesize answers using an LLM

**Key Feature**: Incremental processing - only new files are embedded and saved.

## Setup and Dependencies

In [1]:
# Install required packages
%pip install ollama faiss-cpu nltk numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 9.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 3.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [faiss-cpu]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import ollama
import numpy as np
import faiss
import nltk
import json
from pathlib import Path
from typing import List, Dict, Tuple
from dataclasses import dataclass, asdict

# Download NLTK data for sentence tokenization
nltk.download()

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


True

## Setup

Make sure you have Ollama installed with these models:
```bash
ollama pull qwen3-embedding:8b
ollama pull llama3:8b
```

Smaller models:
```bash
ollama pull qwen3-embedding:0.6b
ollama pull llama3.2:3b
```

In [2]:
@dataclass
class SentenceWithSource:
    """Container for a sentence with its source information"""
    text: str
    file_path: str
    file_title: str  # MD file title for tracking
    line_number: int
    section_header: str = ""

## Helper Functions

In [3]:
# This uses qwen3-embedding:8b model to generate a 4096-dimension embedding for a sentence
def get_embedding(text: str, model: str = "qwen3-embedding:8b") -> np.ndarray:
    """Get embedding vector for text using Ollama
    
    Args:
        text (str): The text to generate an embedding for
        model (str, optional): The model to use. Defaults to "qwen3-embedding:8b".
    
    Returns:
        np.ndarray: The embedding vector
    """
    try:
        response = ollama.embeddings(model=model, prompt=text)
        return np.array(response['embedding'], dtype=np.float32)
    except Exception as e:
        print(f"Error getting embedding: {e}")
        return None

In [4]:
def split_into_sentences(text: str, file_path: str, file_title: str) -> List[SentenceWithSource]:
    """Split text into sentences while tracking source information
    
    Args:
        text (str): The text to split into sentences
        file_path (str): The path to the file the text is from
        file_title (str): The title of the file
    
    Returns:
        List[SentenceWithSource]: A list of sentences with source information
    """
    sentences_with_source = []
    lines = text.split('\n')
    current_section = ""
    
    for line_num, line in enumerate(lines, 1):
        line = line.strip()
        if not line:
            continue
            
        # Track section headers
        if line.startswith('#'):
            current_section = line.strip('#').strip()
            continue
        
        # Skip tables, images, and short lines
        if '|' in line or line.startswith('---') or line.startswith('![]') or line.startswith('Fig.'):
            continue
        
        # Split into sentences
        sentences = nltk.sent_tokenize(line)
        
        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence) > 20:  # Filter short sentences
                sentences_with_source.append(
                    SentenceWithSource(
                        text=sentence,
                        file_path=file_path,
                        file_title=file_title,
                        line_number=line_num,
                        section_header=current_section
                    )
                )
    
    return sentences_with_source

## Storage Configuration

In [6]:
# Storage paths
STORAGE_DIR = Path("vector_store_2")
STORAGE_DIR.mkdir(exist_ok=True)

EMBEDDINGS_FILE = STORAGE_DIR / "embeddings.npy"
FAISS_INDEX_FILE = STORAGE_DIR / "index.faiss"
METADATA_FILE = STORAGE_DIR / "metadata.json"

print(f"Storage directory: {STORAGE_DIR.absolute()}")

Storage directory: /Users/charlesrbrowniii/kokoro/crag-doc-reader/vector_store_2


In [7]:
def load_existing_data():
    """Load existing embeddings, index, and metadata if they exist
    
    Args:
        None
    
    Returns:
        Tuple[List[SentenceWithSource], np.ndarray, faiss.Index, Set[str]]: A tuple containing the sentences, embeddings, index, and processed files
    """
    if METADATA_FILE.exists():
        with open(METADATA_FILE, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        
        embeddings = np.load(EMBEDDINGS_FILE) if EMBEDDINGS_FILE.exists() else None
        index = faiss.read_index(str(FAISS_INDEX_FILE)) if FAISS_INDEX_FILE.exists() else None
        
        # Reconstruct sentences from metadata
        sentences = [
            SentenceWithSource(**item) for item in metadata['sentences']
        ]
        processed_files = set(metadata.get('processed_files', []))
        
        print(f"Loaded {len(sentences)} sentences from {len(processed_files)} files")
        return sentences, embeddings, index, processed_files
    
    return [], None, None, set()

In [23]:
def save_data(sentences: List[SentenceWithSource], embeddings: np.ndarray, 
              index: faiss.Index, processed_files: set, model: str = "qwen3-embedding:8b"):
    """Save embeddings, FAISS index, and metadata
    
    Args:
        sentences (List[SentenceWithSource]): The sentences to save
        embeddings (np.ndarray): The embeddings to save
        index (faiss.Index): The FAISS index to save
        processed_files (set): The processed files to save
    """
    # Save embeddings
    np.save(EMBEDDINGS_FILE, embeddings)
    
    # Save FAISS index
    faiss.write_index(index, str(FAISS_INDEX_FILE))
    
    # Save metadata
    metadata = {
        'sentences': [asdict(s) for s in sentences],
        'processed_files': list(processed_files),
        'embedding_model': model,
        'dimension': embeddings.shape[1],
        'total_sentences': len(sentences)
    }
    
    with open(METADATA_FILE, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"Saved {len(sentences)} sentences, embeddings, and FAISS index")

## Process MD Files (Incremental)

In [24]:
# Load existing data
sentences, embeddings, index, processed_files = load_existing_data()

print(f"Previously processed files: {processed_files}")

Loaded 99 sentences from 1 files
Previously processed files: {'Dahl-1972-Ecology.reef.algae.AS'}


In [25]:
# Specify MD file to process
md_file_path = r"output/Dahl-1972-Ecology.reef.algae.AS/Dahl-1972-Ecology.reef.algae.AS.md"
file_path = Path(md_file_path).resolve()
file_title = file_path.stem  # Use filename without extension as title

print(f"File: {file_path}")
print(f"Title: {file_title}")

# Check if already processed
if file_title in processed_files:
    print(f"⚠️ '{file_title}' already processed. Skipping...")
else:
    print(f"✓ New file - will process")

File: /Users/charlesrbrowniii/kokoro/crag-doc-reader/output/Dahl-1972-Ecology.reef.algae.AS/Dahl-1972-Ecology.reef.algae.AS.md
Title: Dahl-1972-Ecology.reef.algae.AS
⚠️ 'Dahl-1972-Ecology.reef.algae.AS' already processed. Skipping...


In [26]:
# Process new file if not already done
if file_title not in processed_files:
    # Read file
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Extract sentences
    new_sentences = split_into_sentences(content, str(file_path), file_title)
    print(f"Extracted {len(new_sentences)} sentences")
    
    # Generate embeddings
    print("Generating embeddings...")
    new_embeddings = []
    for i, sentence in enumerate(new_sentences):
        if i % 10 == 0:
            print(f"  {i+1}/{len(new_sentences)}")
        
        embedding = get_embedding(sentence.text, model="qwen3-embedding:0.6b")
        if embedding is not None:
            new_embeddings.append(embedding)
        else:
            # Fallback to zero vector
            print(f"  {i+1}/{len(new_sentences)}: Failed to generate embedding")
            new_embeddings.append(np.zeros(1024, dtype=np.float32))
    
    new_embeddings = np.vstack(new_embeddings)
    
    # Normalize for cosine similarity
    faiss.normalize_L2(new_embeddings)
    
    # Merge with existing data
    if embeddings is not None:
        embeddings = np.vstack([embeddings, new_embeddings])
        sentences.extend(new_sentences)
    else:
        embeddings = new_embeddings
        sentences = new_sentences
    
    # Rebuild FAISS index
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    
    # Mark as processed
    processed_files.add(file_title)
    
    # Save everything
    save_data(sentences, embeddings, index, processed_files)
    
    print(f"✓ Processed and saved. Total: {len(sentences)} sentences from {len(processed_files)} files")

## Semantic Search

In [13]:
def search(query: str, top_k: int = 5) -> List[Tuple[SentenceWithSource, float]]:
    """Search for most similar sentences to query
    
    Args:
        query (str): The query to search for
        top_k (int, optional): The number of results to return. Defaults to 5.
    
    Returns:
        List[Tuple[SentenceWithSource, float]]: A list of tuples containing the most similar sentences and their scores
    """
    if index is None:
        raise ValueError("No index loaded. Process files first.")
    
    # Get and normalize query embedding
    query_embedding = get_embedding(query, model="qwen3-embedding:0.6b")
    if query_embedding is None:
        return []
    
    query_embedding = query_embedding.reshape(1, -1)
    faiss.normalize_L2(query_embedding)
    
    # Search
    scores, indices = index.search(query_embedding, top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(sentences):
            results.append((sentences[idx], float(score)))
    
    return results

In [27]:
# Test search
query = "What is the definition of an algal turf?"
results = search(query, top_k=5)

print(f"Query: {query}\n")
print("Top 5 Results:")
print("=" * 60)

for i, (sentence, score) in enumerate(results, 1):
    print(f"{i}. Score: {score:.4f} | File: {sentence.file_title}")
    print(f"   Section: {sentence.section_header}")
    print(f"   Text: {sentence.text[:120]}...")
    print()

Query: What is the definition of an algal turf?

Top 5 Results:
1. Score: 0.7717 | File: Dahl-1972-Ecology.reef.algae.AS
   Section: INTRODUCTION
   Text: Algal turfs can be defined as relatively dense associations of one or more species of filamentous or foliose algae of sm...

2. Score: 0.6965 | File: Dahl-1972-Ecology.reef.algae.AS
   Section: RESULTS
   Text: In terms of percent coverage, turfs are frequently the most significant algal component in shallow reef areas....

3. Score: 0.6920 | File: Dahl-1972-Ecology.reef.algae.AS
   Section: Discussion
   Text: The significance of algal turfs for the whole reef ecosystem is similarly open to conjecture....

4. Score: 0.6919 | File: Dahl-1972-Ecology.reef.algae.AS
   Section: RESULTS
   Text: Algal turfs cover most of the available solid surface in the reef moat with fleshy crusts also common on coral fragments...

5. Score: 0.6898 | File: Dahl-1972-Ecology.reef.algae.AS
   Section: Discussion
   Text: How do such complex associations

In [28]:
def synthesize_answer(query: str, results: List[Tuple[SentenceWithSource, float]], 
                      model: str = "llama3:8b") -> str:
    """Use LLM to synthesize answer from relevant sentences
    
    Args:
        query (str): The query to answer
        results (List[Tuple[SentenceWithSource, float]]): The results to use
        model (str, optional): The model to use. Defaults to "llama3:8b".
    
    Returns:
        str: The answer
    """
    context = "\n\n".join([
        f"[{sent.file_title}] {sent.section_header}\n{sent.text}"
        for sent, score in results
    ])
    
    prompt = f"""Based on the following context, answer the question. Be specific and cite sources.

Context:
{context}

Question: {query}

Answer:"""
    
    try:
        response = ollama.generate(model=model, prompt=prompt)
        return response['response']
    except Exception as e:
        return f"Error: {e}"

In [29]:
# Generate answer using LLM
query = "What is the definition of an algal turf?"
results = search(query, top_k=5)
model = "llama3.2:3b"

print(f"Query: {query}\n")
answer = synthesize_answer(query, results, model)

print("Answer:")
print("=" * 60)
print(answer)
print("=" * 60)

Query: What is the definition of an algal turf?

Answer:
According to Dahl (1972), an algal turf can be defined as a relatively dense association of one or more species of filamentous or foliose algae that attains a height or thickness of 1 to 30 mm [Dahl-1972-Ecology.reef.algae.AS]. This definition specifically highlights the key characteristics of an algal turf, including its density and size range.

It's worth noting that Dahl's definition is based on his work on coral reef algae, and it provides a clear understanding of what constitutes an algal turf in this context.


In [30]:
# Test search
query = "What is the definition of an algal turf?"
results = search(query, top_k=5)

print(f"Query: {query}\n")
print("Top 5 Results:")
print("=" * 60)

for i, (sentence, score) in enumerate(results, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   File: {sentence.file_title}")
    print(f"   Section: {sentence.section_header}")
    print(f"   Text: {sentence.text}...")
    print()

Query: What is the definition of an algal turf?

Top 5 Results:
1. Score: 0.7717
   File: Dahl-1972-Ecology.reef.algae.AS
   Section: INTRODUCTION
   Text: Algal turfs can be defined as relatively dense associations of one or more species of filamentous or foliose algae of small stature, attaining a height or thickness of 1 to 30 mm....

2. Score: 0.6965
   File: Dahl-1972-Ecology.reef.algae.AS
   Section: RESULTS
   Text: In terms of percent coverage, turfs are frequently the most significant algal component in shallow reef areas....

3. Score: 0.6920
   File: Dahl-1972-Ecology.reef.algae.AS
   Section: Discussion
   Text: The significance of algal turfs for the whole reef ecosystem is similarly open to conjecture....

4. Score: 0.6919
   File: Dahl-1972-Ecology.reef.algae.AS
   Section: RESULTS
   Text: Algal turfs cover most of the available solid surface in the reef moat with fleshy crusts also common on coral fragments....

5. Score: 0.6898
   File: Dahl-1972-Ecology.reef.algae.AS